# Soma — free Colab (T4) run

**What this does:** runs Meta's public **TRIBE v2** brain-encoding model on a folder of
video clips and caches, per clip, the predicted cortical activation
(`preds_<id>.npy`, shape `(n_seconds, 20484)`) plus a per-second arc
(`arc_<id>.csv` + `arc_<id>.json`) that is drop-in compatible with
`tvsum_prep.py` / `honest_corr_timeseries.py` / the demo player.

**Config:** audio + video only (the text / LLaMA path is OFF) — the verified config that
ran on a free T4 before.

**Honesty (do not edit these out):**
- Step 1 — *video → brain activation* — is the only validated link (TRIBE, benchmarked vs real fMRI).
- The activation → attention/engagement reading is **our downstream hypothesis, not a result.**
- `preds` are **z-scored, SIGNED BOLD (~[-1, +1])**, NOT 0..1 probabilities. Cell 4 prints
  the real distribution once so you can confirm this — never assume it.
- Nothing here is trained on our side; this is inference only. No fabricated numbers.

**Two ways to feed it data — pick ONE:**
- **Path A — simplest, heaviest on GPU:** Cell 2 downloads all of TVSum into Colab and does everything here.
- **Path B — recommended, least GPU:** run `tvsum_trim.py` on your laptop (it trims each video to ~2 min + downscales, all free), drop the small clips in Google Drive, and use **Cell 2B**. Colab does *only* the brain-math, and results cache to Drive so runs **resume** across disconnects — a wiped machine or a hit quota never costs you a redo.

**Run order:** Cell 1 (install → **restart runtime**) → **Cell 2 _or_ Cell 2B** → Cell 3 (load model) → Cell 4 (extract) → Cell 5 (download).


## Cell 1 — install pinned deps  (then Runtime ▸ Restart session)

In [ ]:
# === Cell 1: install pinned deps for the verified audio+video TRIBE v2 stack =====
# After this finishes it STOPS with a SystemExit telling you to RESTART THE RUNTIME
# (Runtime > Restart session), then run again starting at Cell 2 — skip Cell 1 on the
# second pass. The restart is REQUIRED: NumPy/SciPy are force-pinned below into TRIBE's
# known-good range and Colab must reload them. Do not skip it.
import os, sys, subprocess
from importlib import metadata

# facebook/tribev2 is ~1 GB; a cold T4 download can be slow — give HF room to breathe.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

def _mm(v):
    out = []
    for tok in v.split("."):
        d = "".join(c for c in tok if c.isdigit())
        if not d:
            break
        out.append(int(d))
        if len(out) == 2:
            break
    while len(out) < 2:
        out.append(0)
    return tuple(out)

need = False
try:
    if _mm(metadata.version("numpy")) >= (2, 1):   # TRIBE needs numpy>=1.26,<2.1
        need = True
except Exception:
    need = True
try:
    if not ((1, 13) <= _mm(metadata.version("scipy")) < (1, 16)):
        need = True
except Exception:
    need = True
for dist in ["tribev2", "neuralset", "exca", "nibabel", "nilearn"]:
    try:
        metadata.version(dist)
    except metadata.PackageNotFoundError:
        need = True

if need:
    print("Installing Soma / TRIBE deps (a few minutes on a cold T4)...")
    # Full notebook stack first...
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
         "exca", "yt-dlp", "pillow", "pandas", "matplotlib", "moviepy"],
        check=True,
    )
    # ...then force NumPy/SciPy back to TRIBE's compiled-dep range.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--force-reinstall",
         "numpy>=1.26.4,<2.1", "scipy>=1.13,<1.16"],
        check=True,
    )
    # keep torchaudio in LOCKSTEP with torch (the tribev2 install can pull a
    # newer torchaudio -> "undefined symbol: aoti_torch_abi_version" crash in Cell 4).
    # Read the freshly-installed torch from a subprocess (this process has a stale one).
    _tv = subprocess.run([sys.executable, "-c",
        "import torch;print(torch.__version__.split('+')[0])"],
        capture_output=True, text=True).stdout.strip()
    _cu = subprocess.run([sys.executable, "-c",
        "import torch;print('cu'+(torch.version.cuda or '12.1').replace('.',''))"],
        capture_output=True, text=True).stdout.strip()
    if _tv:
        # CLEAN reinstall (uninstall first): a --force-reinstall can leave mixed
        # torchaudio files -> "cannot import name '_init_sox'". Remove, then install one copy.
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=False)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=False)
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             f"torchaudio=={_tv}", "--index-url", f"https://download.pytorch.org/whl/{_cu}"],
            check=False,
        )
    raise SystemExit(
        "Install complete. Now: Runtime > Restart session, then run from Cell 2 (skip Cell 1)."
    )

print("Deps already present.")
print("  NumPy:", metadata.version("numpy"), " SciPy:", metadata.version("scipy"))
print("  HF_HUB_DOWNLOAD_TIMEOUT =", os.environ["HF_HUB_DOWNLOAD_TIMEOUT"])


## Cell 2 — get the data (TVSum50, no Google Drive needed)

Downloads the official TVSum50 archive **straight into this Colab runtime**, extracts it,
and points the pipeline at a subset of videos. It also tucks `ydata-tvsum50.mat` into the
results folder so it ends up in the download zip (you need it for `tvsum_prep.py` on your
laptop). Using your **own ad clips** instead? Skip this and run **Cell 2-ALT** below.

> TVSum videos are full-length (1–11 min each), so extraction is where the time goes.
> Start with a small `N_CLIPS`; longer videos = more timepoints = a *stronger* per-video test.

In [ ]:
# === Cell 2: download + stage TVSum50 (no Drive needed) =========================
import subprocess, shutil
from pathlib import Path

N_CLIPS = 3    # first run: prove the pipeline end-to-end fast, then raise to 10+ for a real batch

WORK = Path("/content/tvsum"); WORK.mkdir(exist_ok=True)
TGZ  = WORK / "tvsum50_ver_1_1.tgz"
URL  = "http://people.csail.mit.edu/yalesong/tvsum/tvsum50_ver_1_1.tgz"

if not TGZ.exists():
    print("Downloading TVSum50 (641 MB) ...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(TGZ), URL], check=True)
if not any(WORK.rglob("*.mp4")):
    print("Extracting ...")
    subprocess.run(["tar", "xzf", str(TGZ), "-C", str(WORK)], check=True)
# TVSum nests the real data inside .zip files — unzip those too
for _z in sorted(WORK.rglob("*.zip")):
    subprocess.run(["unzip", "-o", "-q", str(_z), "-d", str(_z.parent)], check=False)

# locate videos under ANY common extension (TVSum may not be .mp4); if none,
# print the extracted tree so the layout is obvious instead of a blind crash.
import os, collections
VIDEO_EXTS = (".mp4",".webm",".mkv",".avi",".mov",".m4v",".flv",".mpg",".mpeg")
mp4s = sorted(p for p in WORK.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
mats = sorted(WORK.rglob("ydata-tvsum50.mat"))
if not mp4s:
    print("No video files found. What actually extracted:")
    for r,_,f in os.walk(WORK):
        d = r.replace(str(WORK),"").count(os.sep)
        if d <= 2: print("  "*d + os.path.basename(r) + f"/  ({len(f)} files)")
    exts = collections.Counter(os.path.splitext(f)[1].lower()
                               for r,_,fs in os.walk(WORK) for f in fs)
    print("file extensions present:", exts.most_common(20))
    raise SystemExit("No videos in the archive — paste the tree above to Claude "
                     "(we may need yt-dlp to fetch them by YouTube ID).")
VIDEO_SRC = mp4s[0].parent
MAT = mats[0] if mats else None

# ---- config the rest of the notebook reads (drop-in for the Drive cell) ---------
CLIP_DIR = WORK / "clips"; CLIP_DIR.mkdir(exist_ok=True)
for v in mp4s[:N_CLIPS]:
    dst = CLIP_DIR / v.name
    if not dst.exists():
        shutil.copy(v, dst)

OUT_DIR = WORK / "arcs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
GLOB = "*" + mp4s[0].suffix.lower()
ROI_MASK_PATH = None   # set to a build_roi_mask.py mask (upload it) to ALSO get roi_mag
DEMO_FEATURE = "roi"   # falls back to 'global' automatically when no ROI mask is given

# stash the annotations so Cell 5's zip carries them back to your laptop
if MAT is not None:
    shutil.copy(MAT, OUT_DIR / "ydata-tvsum50.mat")

clips = sorted(CLIP_DIR.glob(GLOB))
print(f"\nTVSum ready:")
print(f"  videos in archive : {len(mp4s)}  (processing {len(clips)} this run — edit N_CLIPS)")
print(f"  CLIP_DIR : {CLIP_DIR}")
print(f"  OUT_DIR  : {OUT_DIR}")
print(f"  .mat     : {MAT}  (copied into OUT_DIR -> lands in the results zip)")
for c in clips:
    print("   -", c.name)

## Cell 2B — LIGHT path: trimmed clips from Drive (recommended, least GPU)

**Use this INSTEAD of Cell 2** once you've prepped clips on your laptop with
`tvsum_trim.py` (it trims to ~2 min + downscales) — or for your own ad clips.
Colab does only the GPU brain-math: no 641 MB download, no extraction, no ffmpeg on
the GPU clock. And because `OUT_DIR` lives on Drive, **finished clips persist** — a
disconnect or a hit quota resumes instead of restarting.

In [ ]:
# === Cell 2B — LIGHT path: pre-trimmed clips from Drive (RECOMMENDED) ============
# Pair with tvsum_trim.py on your laptop: it downloads TVSum, trims each video to its
# first ~2 min + downscales, and you drop the small clips in the Drive folder below.
#
# RESUME: OUT_DIR is on Drive, so finished clips survive disconnects. Re-running Cell 4
# SKIPS clips already done -> 3 today + 3 tomorrow just stack up, and a mid-batch
# disconnect never costs a redo. This is what stretches the free quota furthest.
# (SKIP Cell 2 above if you use this cell.)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

# ---- EDIT THESE to match your Drive layout --------------------------------------
CLIP_DIR = Path("/content/drive/MyDrive/soma/clips")   # trimmed *.mp4 from tvsum_trim.py
OUT_DIR  = Path("/content/drive/MyDrive/soma/arcs")     # results persist here (resume!)
GLOB     = "*.mp4"                                       # which files to process

# OPTIONAL a-priori ROI mask (boolean .npy, length 20484) from build_roi_mask.py.
# Upload it to Drive and set the path to ALSO get roi_mag (the a-priori ROI test).
# Leave None for whole-cortex GLOBAL only (the documented likely-null baseline).
ROI_MASK_PATH = None    # e.g. Path("/content/drive/MyDrive/soma/roi_mask_dmn.npy")
DEMO_FEATURE  = "roi"   # which feature drives arc_<id>.json; falls back to "global"
                        # automatically when no ROI mask is supplied.
# ---------------------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
clips = sorted(CLIP_DIR.glob(GLOB))
done  = {p.stem[len("preds_"):] for p in OUT_DIR.glob("preds_*.npy")}
todo  = [c for c in clips if c.stem not in done]
print(f"clip dir : {CLIP_DIR}")
print(f"          {len(clips)} clip(s) match {GLOB!r}  |  {len(done)} already done -> {len(todo)} to run")
for c in clips[:12]:
    print(f"   - [{'done' if c.stem in done else 'todo'}] {c.name}")
if len(clips) > 12:
    print(f"   ... (+{len(clips) - 12} more)")
if not clips:
    print("!! No clips found — run tvsum_trim.py on your laptop and upload the folder to")
    print("  ", CLIP_DIR)
print(f"out dir  : {OUT_DIR}  (on Drive -> results survive disconnects)")
print(f"roi mask : {ROI_MASK_PATH}")


## Cell 3 — load TRIBE v2 (audio + video, the verified config)

In [ ]:
# === Cell 3: load the model ======================================================
import os
# Re-set here too: env vars are cleared by the runtime restart after Cell 1.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")

from pathlib import Path
from tribev2.demo_utils import TribeModel

CACHE_FOLDER = Path("./cache")
print("Loading facebook/tribev2 (audio+video) — first run downloads ~1 GB to ./cache ...")
model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={"data": {"features_to_use": ["audio", "video"]}},
)
print("Model loaded.  features_to_use = ['audio', 'video']  (text / LLaMA path OFF).")


## Cell 4 — batch extract

Per clip: build events → `model.predict` → save `preds_<id>.npy`, then reduce to
`arc_<id>.csv` (`t_sec, global_mag[, roi_mag]`) + `arc_<id>.json`.
The functions below mirror `batch_extract.py` so the outputs are byte-compatible with the
CPU analysis + demo. Skips already-cached clips; one bad clip cannot kill the batch.

In [ ]:
# === Cell 4: predict -> preds_<id>.npy + arc_<id>.csv + arc_<id>.json ============
# Mirrors batch_extract.py (the GPU-box runbook) so Colab output is drop-in compatible
# with tvsum_prep.py / honest_corr_timeseries.py / the demo player.
import json, traceback
import numpy as np
import pandas as pd
from neuralset.events.utils import standardize_events
from neuralset.events.transforms import ExtractAudioFromVideo, ChunkEvents

# --- guard: Cell 2 (or 2B) must have run to define the paths this cell needs
_need = [v for v in ("CLIP_DIR", "OUT_DIR", "GLOB", "ROI_MASK_PATH", "DEMO_FEATURE",
                     "clips") if v not in globals()]
if _need:
    raise SystemExit("→ Run Cell 2 (get TVSum) first — or Cell 2B for clips from "
                     "Drive — it sets the paths this cell needs. Missing: " + ", ".join(_need))
if not clips:
    raise SystemExit(f"0 clips found in {CLIP_DIR} (GLOB={GLOB!r}). If you meant TVSum, run "
                     "Cell 2 (the download cell); for the light path run tvsum_trim.py on your "
                     "laptop and upload the clips to the Cell 2B Drive folder.")


def build_events(video_path):
    """The working manual event-build path from the notebook (no transcription)."""
    transforms = [
        ExtractAudioFromVideo(),
        ChunkEvents(event_type_to_chunk="Audio", max_duration=60, min_duration=30),
        ChunkEvents(event_type_to_chunk="Video", max_duration=60, min_duration=30),
    ]
    initial = {"type": "Video", "filepath": str(video_path), "start": 0,
               "timeline": "default", "subject": "default"}
    df = standardize_events(pd.DataFrame([initial]))
    for t in transforms:
        df = t(df)
    return standardize_events(df)


def describe_preds(preds, tag=""):
    """Print the distribution ONCE so the z-scored/signed scale is confirmed, not assumed."""
    p = np.asarray(preds, float)
    pct = np.percentile(p, [1, 50, 90, 99])
    print(f"  [preds {tag}] shape={p.shape} min={p.min():.3f} max={p.max():.3f} "
          f"mean={p.mean():.3f} std={p.std():.3f}")
    print(f"           pct(1/50/90/99)={pct[0]:.3f}/{pct[1]:.3f}/{pct[2]:.3f}/{pct[3]:.3f}"
          f"  frac<0={np.mean(p < 0):.2%}  frac>0.6={np.mean(p > 0.6):.2%}")
    if p.min() >= 0 and p.max() <= 1:
        print("           WARNING: values look bounded [0,1] - verify this really is "
              "z-scored BOLD, not a probability, before trusting any threshold.")


def arc_from_preds(preds, roi_mask=None):
    """(T, 20484) -> per-second magnitude scalars. |preds| avoids the old threshold bug."""
    p = np.abs(np.asarray(preds, float))
    global_mag = p.mean(axis=1)
    roi_mag = None
    if roi_mask is not None:
        m = np.asarray(roi_mask, bool)
        if m.shape[0] != p.shape[1]:
            raise ValueError(f"ROI mask length {m.shape[0]} != n_vertices {p.shape[1]}")
        roi_mag = p[:, m].mean(axis=1)
    return global_mag, roi_mag


def detect_weak_spots(arc, fps=1.0, min_len_sec=3, drop_pctl=25):
    """A weak spot = a sustained run in the arc's own bottom quartile. A PREDICTION, not behavior."""
    a = np.asarray(arc, float)
    if len(a) < 5:
        return []
    k = max(1, int(round(fps)))
    sm = np.convolve(a, np.ones(k) / k, mode="same")
    thr = np.percentile(sm, drop_pctl)
    low = sm <= thr
    spots, i, n = [], 0, len(a)
    min_len = int(round(min_len_sec * fps))
    while i < n:
        if low[i]:
            j = i
            while j < n and low[j]:
                j += 1
            if (j - i) >= min_len:
                spots.append({"start": round(i / fps, 2), "end": round(j / fps, 2),
                              "label": f"predicted dip - {round((j - i) / fps)}s "
                                       f"low-activation stretch"})
            i = j
        else:
            i += 1
    return spots


def write_demo_json(out_dir, vid, arc, weak_spots, feature_name, fps=1.0):
    a = np.asarray(arc, float)
    lo, hi = np.percentile(a, 2), np.percentile(a, 98)
    norm = np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
    payload = {
        "video_id": vid, "fps_arc": fps, "duration_sec": round(len(a) / fps, 2),
        "feature": feature_name, "precomputed": True,
        "claim": {
            "validated": "video -> brain activation (Meta TRIBE v2, public model, "
                         "benchmarked vs real fMRI)",
            "hypothesis": "activation -> engagement (our downstream inference; "
                          "tested, not asserted)",
        },
        "timestamps": [round(i / fps, 2) for i in range(len(a))],
        "activation": [round(float(x), 4) for x in norm],
        "weak_spots": weak_spots,
    }
    path = out_dir / f"arc_{vid}.json"
    path.write_text(json.dumps(payload, indent=2))
    return path


# --- load ROI mask if one was supplied in Cell 2 ---------------------------------
roi_mask = None
if ROI_MASK_PATH is not None:
    roi_mask = np.load(ROI_MASK_PATH)
    print(f"[roi] mask {Path(ROI_MASK_PATH).name}: "
          f"{int(np.asarray(roi_mask, bool).sum())}/{roi_mask.shape[0]} vertices")
else:
    print("[note] no ROI mask -> GLOBAL only. Build one with build_roi_mask.py + set "
          "ROI_MASK_PATH in Cell 2 to run the a-priori ROI test.")

printed_scale = False
n_ok, n_fail = 0, 0
for vp in clips:
    vid = vp.stem
    npy = OUT_DIR / f"preds_{vid}.npy"
    try:
        if npy.exists():
            print(f"[skip-cached] {vid}")
            preds = np.load(npy)
        else:
            print(f"[predict] {vid} ...")
            events = build_events(vp)
            preds, _segments = model.predict(events=events)
            preds = np.asarray(preds, float)
            np.save(npy, preds.astype(np.float32))  # float32 halves Drive size; z-scored BOLD needs no more precision
        if not printed_scale:
            describe_preds(preds, tag=vid)      # confirm z-scored/signed scale ONCE
            printed_scale = True

        global_mag, roi_mag = arc_from_preds(preds, roi_mask)
        T = len(global_mag)
        has_roi = roi_mag is not None
        with open(OUT_DIR / f"arc_{vid}.csv", "w") as f:
            f.write("t_sec,global_mag" + (",roi_mag\n" if has_roi else "\n"))
            for t in range(T):
                if has_roi:
                    f.write(f"{t},{global_mag[t]:.6f},{roi_mag[t]:.6f}\n")
                else:
                    f.write(f"{t},{global_mag[t]:.6f}\n")

        use_roi = (DEMO_FEATURE == "roi" and roi_mag is not None)
        demo_arc = roi_mag if use_roi else global_mag
        feat = "roi" if use_roi else "global"    # label the arc with what it actually is
        spots = detect_weak_spots(demo_arc, fps=1.0)
        write_demo_json(OUT_DIR, vid, demo_arc, spots, feature_name=feat)
        print(f"  ok: T={T}s  weak_spots={len(spots)}  feature={feat}")
        n_ok += 1
    except Exception as e:   # one bad clip must not kill the batch
        n_fail += 1
        print(f"  [ERROR] {vid}: {e!r} - skipping, batch continues")
        traceback.print_exc()

print(f"\n[batch done] ok={n_ok} failed={n_fail}  ->  {OUT_DIR}")
print("Reminder: preds are z-scored SIGNED BOLD (~[-1,1]), NOT probabilities - "
      "confirm from the [preds ...] distribution printed above.")


## Cell 5 — zip the results and download

In [ ]:
# === Cell 5: zip + download ======================================================
import shutil
from pathlib import Path

archive = shutil.make_archive("/content/soma_arcs", "zip", root_dir=str(OUT_DIR))
print("zipped ->", archive)

try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("Auto-download unavailable:", e)
    print(f"Your results are already saved in Drive at: {OUT_DIR}")
